In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
url = "https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population"

response = requests.get(
    url,
    headers={"User-Agent": "CodeAlphaWebScrapingProject/1.0"},
    timeout=30
)

print("Status code:", response.status_code)
print("Page retrieved:", response.url)

Status code: 200
Page retrieved: https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population


In [3]:
soup = BeautifulSoup(response.text, "html.parser")

tables = soup.find_all("table")

print("Number of tables found:", len(tables))

Number of tables found: 3


In [4]:
for i, table in enumerate(tables):
    print(f"\nTable {i}")
    print(table.get_text(" ", strip=True)[:300])


Table 0
List of countries and territories by total population Location Population % of world Date Source (official or from the United Nations ) Notes World 8,232,000,000 100% 13 Jun 2025 UN projection [ 1 ] [ 3 ] India 1,429,404,000 17.3% 1 Jul 2026 Official projection [ 4 ] [ b ] China 1,404,890,000 17.0% 

Table 1
v t e Lists of countries by population statistics Global Current population United Nations Demographics of the world Continents/subregions Africa Antarctica Asia Europe North America Caribbean Oceania South America Intercontinental Americas Arab world Commonwealth of Nations Eurasia European Union I

Table 2
v t e World Afghanistan Albania Algeria Andorra Angola Antigua and Barbuda Argentina Armenia Australia Austria Azerbaijan Bahamas Bahrain Bangladesh Barbados Belarus Belgium Belize Benin Bhutan Bolivia Bosnia and Herzegovina Botswana Brazil Brunei Bulgaria Burkina Faso Burundi Cambodia Cameroon Cana


In [5]:
df = pd.read_html(str(tables[0]))[0]

print("Dataset shape:", df.shape)
df.head()

/tmp/ipykernel_1041/2178128701.py:1: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(str(tables[0]))[0]


Dataset shape: (241, 6)


,Location,Population,% of world,Date,Source (official or from the United Nations),Notes
0,World,8.232000e+09,100%,13 Jun 2025,UN projection[1][3],NaN
1,India,1.429404e+09,17.3%,1 Jul 2026,Official projection[4],[b]
2,China,1.404890e+09,17.0%,31 Dec 2025,Official estimate[5],[c]
3,United States,3.417849e+08,4.1%,1 Jul 2025,Official estimate[6],[d]
4,Indonesia,2.883151e+08,3.5%,31 Dec 2025,National annual projection[7],NaN


In [6]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset dimensions:")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Column names:
['Location', 'Population', '% of world', 'Date', 'Source (official or from the United Nations)', 'Notes']

Dataset dimensions:
Rows: 241
Columns: 6


In [7]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

Missing values per column:
Location                                          1
Population                                        1
% of world                                        1
Date                                              1
Source (official or from the United Nations)      1
Notes                                           208
dtype: int64

Duplicate rows: 0

Data types:
Location                                         object
Population                                      float64
% of world                                       object
Date                                             object
Source (official or from the United Nations)     object
Notes                                            object
dtype: object


In [8]:
df[df["Location"].isnull()]

,Location,Population,% of world,Date,Source (official or from the United Nations),Notes
235,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df.to_csv("world_population_raw.csv", index=False)

print("CSV file saved successfully!")

CSV file saved successfully!


In [10]:
from google.colab import files
files.download("world_population_raw.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
df = df.dropna(subset=["Location"]).copy()
df = df.reset_index(drop=True)

print("Updated dataset shape:", df.shape)
print("Missing locations:", df["Location"].isnull().sum())

Updated dataset shape: (240, 6)
Missing locations: 0


In [13]:
df[df["Population"].isnull()]

,Location,Population,% of world,Date,Source (official or from the United Nations),Notes


In [14]:
print("Missing population values:", df["Population"].isnull().sum())

Missing population values: 0


In [15]:
print("Population data type:", df["Population"].dtype)

print("\nFirst 10 population values:")
print(df["Population"].head(10).to_list())

Population data type: float64

First 10 population values:
[8232000000.0, 1429404000.0, 1404890000.0, 341784857.0, 288315089.0, 241499431.0, 223800000.0, 214211951.0, 169828911.0, 146028325.0]


In [16]:
print(
    "Values with decimal fractions:",
    (df["Population"] % 1 != 0).sum()
)

Values with decimal fractions: 0


In [17]:
df["Population"] = df["Population"].astype("Int64")

print("Population data type:", df["Population"].dtype)
print(df["Population"].head(10).to_list())

Population data type: Int64
[8232000000, 1429404000, 1404890000, 341784857, 288315089, 241499431, 223800000, 214211951, 169828911, 146028325]


In [18]:
print("Data type:", df["% of world"].dtype)

print("\nFirst 10 values:")
print(df["% of world"].head(10).to_list())

Data type: object

First 10 values:
['100%', '17.3%', '17.0%', '4.1%', '3.5%', '2.9%', '2.7%', '2.6%', '2.1%', '1.8%']


In [19]:
df["% of world"] = (
    df["% of world"]
    .str.replace("%", "", regex=False)
    .astype(float)
)

print("New data type:", df["% of world"].dtype)
print(df["% of world"].head(10).to_list())

New data type: float64
[100.0, 17.3, 17.0, 4.1, 3.5, 2.9, 2.7, 2.6, 2.1, 1.8]


In [20]:
print("Data type:", df["Date"].dtype)

print("\nFirst 15 date values:")
print(df["Date"].head(15).to_list())

Data type: object

First 15 date values:
['13 Jun 2025', '1 Jul 2026', '31 Dec 2025', '1 Jul 2025', '31 Dec 2025', '1 Mar 2023', '1 Jul 2023', '1 Jul 2026', '14 Jun 2022', '1 Jan 2025', '30 Jun 2026', '1 Oct 2026', '1 Jul 2026', '1 Jul 2026', '1 Jul 2025']


In [21]:
df["Date"] = pd.to_datetime(
    df["Date"],
    format="%d %b %Y",
    errors="coerce"
)

print("New data type:", df["Date"].dtype)
print("\nFirst 15 dates:")
print(df["Date"].head(15).to_list())

New data type: datetime64[ns]

First 15 dates:
[Timestamp('2025-06-13 00:00:00'), Timestamp('2026-07-01 00:00:00'), Timestamp('2025-12-31 00:00:00'), Timestamp('2025-07-01 00:00:00'), Timestamp('2025-12-31 00:00:00'), Timestamp('2023-03-01 00:00:00'), Timestamp('2023-07-01 00:00:00'), Timestamp('2026-07-01 00:00:00'), Timestamp('2022-06-14 00:00:00'), Timestamp('2025-01-01 00:00:00'), Timestamp('2026-06-30 00:00:00'), Timestamp('2026-10-01 00:00:00'), Timestamp('2026-07-01 00:00:00'), Timestamp('2026-07-01 00:00:00'), Timestamp('2025-07-01 00:00:00')]


In [22]:
print("Missing dates:", df["Date"].isnull().sum())

Missing dates: 10


In [23]:
df.loc[df["Date"].isnull(), ["Location", "Date", "Source (official or from the United Nations)"]]

,Location,Date,Source (official or from the United Nations)
20,Thailand,NaT,Preliminary census results[24]
117,Lebanon,NaT,Official estimate[128]
148,Lesotho,NaT,Official survey[163]
163,Comoros,NaT,Official estimate[178]
165,Bhutan,NaT,National annual projection[180]
186,São Tomé and Príncipe,NaT,Preliminary census result[200]
195,Grenada,NaT,Preliminary Census results[208]
204,Dominica,NaT,Official estimate[217]
211,Saint Kitts and Nevis,NaT,2022 census result[224]
239,Pitcairn Islands (UK),NaT,Official estimate[251]


In [24]:
df_original = pd.read_html(str(tables[0]))[0]

df_original.loc[
    df["Date"].isnull(),
    ["Location", "Date"]
]

/tmp/ipykernel_1041/1784561156.py:1: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_original = pd.read_html(str(tables[0]))[0]


IndexingError: Unalignable boolean Series provided as indexer (index of the boolean Series and of the indexed object do not match).

In [25]:
from io import StringIO

df_original = pd.read_html(StringIO(str(tables[0])))[0]

print("Original shape:", df_original.shape)
print("Current shape:", df.shape)

df_original.head()

Original shape: (241, 6)
Current shape: (240, 6)


,Location,Population,% of world,Date,Source (official or from the United Nations),Notes
0,World,8.232000e+09,100%,13 Jun 2025,UN projection[1][3],NaN
1,India,1.429404e+09,17.3%,1 Jul 2026,Official projection[4],[b]
2,China,1.404890e+09,17.0%,31 Dec 2025,Official estimate[5],[c]
3,United States,3.417849e+08,4.1%,1 Jul 2025,Official estimate[6],[d]
4,Indonesia,2.883151e+08,3.5%,31 Dec 2025,National annual projection[7],NaN


In [26]:
# Remove the empty row from the original extraction
df_original_clean = (
    df_original
    .dropna(subset=["Location"])
    .reset_index(drop=True)
)

# Get the locations whose dates failed to convert
bad_locations = df.loc[df["Date"].isnull(), "Location"]

# Retrieve their original date text
df_original_clean.loc[
    df_original_clean["Location"].isin(bad_locations),
    ["Location", "Date"]
]

,Location,Date
20,Thailand,2025
117,Lebanon,2026
148,Lesotho,2024
163,Comoros,2026
165,Bhutan,2026
186,São Tomé and Príncipe,2024
195,Grenada,2021
204,Dominica,2026
211,Saint Kitts and Nevis,2022
239,Pitcairn Islands (UK),2023


In [27]:
df["Date"] = df_original_clean["Date"].values

print(df[["Location", "Date"]].head(10))

        Location         Date
0          World  13 Jun 2025
1          India   1 Jul 2026
2          China  31 Dec 2025
3  United States   1 Jul 2025
4      Indonesia  31 Dec 2025
5       Pakistan   1 Mar 2023
6        Nigeria   1 Jul 2023
7         Brazil   1 Jul 2026
8     Bangladesh  14 Jun 2022
9         Russia   1 Jan 2025


In [28]:
df["Year"] = df["Date"].str.extract(r"(\d{4})").astype("Int64")

print(df[["Location", "Date", "Year"]].head(15))


                            Location         Date  Year
0                              World  13 Jun 2025  2025
1                              India   1 Jul 2026  2026
2                              China  31 Dec 2025  2025
3                      United States   1 Jul 2025  2025
4                          Indonesia  31 Dec 2025  2025
5                           Pakistan   1 Mar 2023  2023
6                            Nigeria   1 Jul 2023  2023
7                             Brazil   1 Jul 2026  2026
8                         Bangladesh  14 Jun 2022  2022
9                             Russia   1 Jan 2025  2025
10                            Mexico  30 Jun 2026  2026
11                             Japan   1 Oct 2026  2026
12  Democratic Republic of the Congo   1 Jul 2026  2026
13                       Philippines   1 Jul 2026  2026
14                          Ethiopia   1 Jul 2025  2025


In [29]:
print("Missing years:", df["Year"].isnull().sum())

print("\nYear data type:", df["Year"].dtype)

print("\nYear range:")
print("Earliest year:", df["Year"].min())
print("Latest year:", df["Year"].max())

Missing years: 0

Year data type: Int64

Year range:
Earliest year: 2020
Latest year: 2026


In [30]:
source_col = "Source (official or from the United Nations)"

print(df[source_col].head(15).to_list())

['UN projection[1][3]', 'Official projection[4]', 'Official estimate[5]', 'Official estimate[6]', 'National annual projection[7]', '2023 census result[8]', 'Official projection[9]', 'Official estimate[10]', '2022 census result[11]', 'Official estimate[13]', 'National quarterly estimate[14]', 'Monthly national estimate[15]', 'UN projection[16]', 'Official projection[17]', 'National annual projection[18]']


In [31]:
source_col = "Source (official or from the United Nations)"

df[source_col] = (
    df[source_col]
    .str.replace(r"\[\d+\]", "", regex=True)
    .str.strip()
)

print(df[source_col].head(15).to_list())

['UN projection', 'Official projection', 'Official estimate', 'Official estimate', 'National annual projection', '2023 census result', 'Official projection', 'Official estimate', '2022 census result', 'Official estimate', 'National quarterly estimate', 'Monthly national estimate', 'UN projection', 'Official projection', 'National annual projection']


In [32]:
print("Non-missing notes:")
print(df["Notes"].dropna().head(20).to_list())

print("\nMissing notes:", df["Notes"].isnull().sum())

Non-missing notes:
['[b]', '[c]', '[d]', '[e]', '[f]', '[g]', '[h]', '[i]', '[j]', '[k]', '[l]', '[m]', '[n]', '[o]', '[p]', '[q]', '[r]', '[s]', '[t]', '[u]']

Missing notes: 207


In [33]:
df = df.drop(columns=["Notes"])

In [34]:
print("Updated shape:", df.shape)
print("\nRemaining columns:")
print(df.columns.tolist())

Updated shape: (240, 6)

Remaining columns:
['Location', 'Population', '% of world', 'Date', 'Source (official or from the United Nations)', 'Year']


In [35]:
# Count duplicate locations
duplicate_locations = df["Location"].duplicated().sum()

print("Number of duplicate locations:", duplicate_locations)

Number of duplicate locations: 0


In [36]:
text_columns = df.select_dtypes(include=["object"]).columns

for column in text_columns:
    print(f"{column}:")
    print("Leading spaces:", df[column].astype("string").str.startswith(" ").sum())
    print("Trailing spaces:", df[column].astype("string").str.endswith(" ").sum())
    print()

Location:
Leading spaces: 0
Trailing spaces: 0

Date:
Leading spaces: 0
Trailing spaces: 0

Source (official or from the United Nations):
Leading spaces: 0
Trailing spaces: 0



In [37]:
print(df.dtypes)

Location                                         object
Population                                        Int64
% of world                                      float64
Date                                             object
Source (official or from the United Nations)     object
Year                                              Int64
dtype: object


In [38]:
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
Location                                        0
Population                                      0
% of world                                      0
Date                                            0
Source (official or from the United Nations)    0
Year                                            0
dtype: int64


In [39]:
# Find rows with zero or negative population
unusual_population = df[df["Population"] <= 0]

print("Number of unusual population values:", len(unusual_population))
display(unusual_population)

Number of unusual population values: 0


,Location,Population,% of world,Date,Source (official or from the United Nations),Year


In [40]:
print("Population summary:")
print(df["Population"].describe())


Population summary:
count               240.0
mean        67940231.8625
std      546131342.071887
min                  35.0
25%              389319.0
50%             5597000.0
75%           23392188.25
max          8232000000.0
Name: Population, dtype: Float64


In [41]:
df.to_csv("world_population_cleaned.csv", index=False)

In [42]:
from google.colab import files
files.download("world_population_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>